### 0. Create Model (same as train)

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import csv

from pathlib import Path
BASE_DIR = Path.cwd().parent
print(f"Base directory: {BASE_DIR}")

Base directory: d:\Myworkplace\Python\violence-movies


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow import keras
from tensorflow.keras.layers import (Input, Conv3D, MaxPooling3D, ZeroPadding3D, Flatten, Dense, LSTM, ConvLSTM2D,
                                     TimeDistributed, Dropout, GlobalAveragePooling2D, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K

def create_C3D_model(summary = False):
    """Creates model object with the sequential API: https://keras.io/models/sequential/

    Parameters
    ----------
    summary : bool
              if True, prints the model summary (default False)

    Returns
    -------
    model : Sequential
            The instantiated model
    """

    model = Sequential()
    input_shape = (16, 112, 112, 3)

    model.add(Conv3D(64, (3, 3, 3), activation='relu',
                     padding='same', name='conv1',
                     input_shape=input_shape))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2),
                           padding='valid', name='pool1'))
    # 2nd layer group
    model.add(Conv3D(128, (3, 3, 3), activation='relu',
                     padding='same', name='conv2'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool2'))
    # 3rd layer group
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3a'))
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3b'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool3'))
    # 4th layer group
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv4a'))
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv4b'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool4'))
    # 5th layer group
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv5a'))
    model.add(Conv3D(512, (3, 3, 3), activation='relu',
                     padding='same', name='conv5b'))
    model.add(ZeroPadding3D(padding=((0, 0), (0, 1), (0, 1)), name='zeropad5'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool5'))
    model.add(Flatten())
    # FC layers group
    model.add(Dense(4096, activation='relu', name='fc6'))
    model.add(Dropout(.5))
    model.add(Dense(4096, activation='relu', name='fc7'))
    model.add(Dropout(.5))
    model.add(Dense(487, activation='softmax', name='fc8'))

    if summary:
      print(model.summary())

    return model

def create_C3D_refined_model(summary = False):
    """

    Parameters
    ----------
    summary : bool
              if True, prints the model summary (default False)

    Returns
    -------
    model : Sequential
            The instantiated model
    """

    model = Sequential()
    input_shape = (16, 112, 112, 3)

    model.add(Conv3D(64, (3, 3, 3), activation='relu',
                     padding='same', name='conv1',
                     input_shape=input_shape))
    model.add(MaxPooling3D(pool_size=(1, 2, 2), strides=(1, 2, 2),
                           padding='valid', name='pool1'))
    # 2nd layer group
    model.add(Conv3D(128, (3, 3, 3), activation='relu',
                     padding='same', name='conv2'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool2'))
    # 3rd layer group
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3a'))
    model.add(Conv3D(256, (3, 3, 3), activation='relu',
                     padding='same', name='conv3b'))
    model.add(MaxPooling3D(pool_size=(2, 2, 2), strides=(2, 2, 2),
                           padding='valid', name='pool3'))

    if summary:
      print(model.summary())

    return model

def getFeatureExtractor(weightsPath, layer, verbose = False):
    """Gets the C3D feature extractor

    Parameters
    ----------
    weightsPath : str
                  Pathname of the weights file for the C3D model.
    layer : str
            Name of the output layer for the feature extractor
    verbose : bool
              if True print debug logs (default True)

    Returns
    -------

    Model : Model class
            Feature extractor

    """

    keras.backend.clear_session()

    base = create_C3D_model(summary=verbose)  # Sequential built with tf.keras layers

    inp = keras.Input(shape=(16, 112, 112, 3), name="c3d_input")
    x = inp
    pool3_tensor = None

    # Build ONE functional graph using the SAME layer objects
    for l in base.layers:
        x = l(x)
        if l.name == layer:
            pool3_tensor = x

    if pool3_tensor is None:
        raise ValueError(f"Layer '{layer}' not found in model. Available: {[l.name for l in base.layers]}")

    # Load weights AFTER the graph exists
    base.load_weights(weightsPath, by_name=True, skip_mismatch=True)

    feat = keras.Model(inputs=inp, outputs=pool3_tensor, name=f"c3d_{layer}")
    return feat

def getC3DCNNModel(verbose=True, freeze_backbone=True, lr=1e-4):
    """Creates the C3D + fully connected layers end-to-end model object with the
    sequential API: https://keras.io/models/sequential/

    Parameters
    ----------
    verbose : bool
              if True prints the model summary (default True)

    Returns
    -------
    model : Sequential
            The instantiated model
    """
    pretrainedModel = getFeatureExtractor(BASE_DIR / "c3d_weight"/ "3dcnn_weights.h5", 'fc6', False)
    for layer in pretrainedModel.layers:
        layer.trainable = not freeze_backbone

    dropout1 = Dropout(.5)(pretrainedModel.output)
    fc7Alt = Dense(512, activation='relu', name='fc7-alt')(dropout1)
    dropout2 = Dropout(.5)(fc7Alt)
    output = Dense(3, activation='softmax')(dropout2)
    model = Model(inputs=pretrainedModel.inputs, outputs=output)
    if verbose:
        model.summary()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    del pretrainedModel

    return model

def getLSTMModel(verbose=True, freeze_backbone=None, lr=1e-4):
    """Creates the ConvLSTM + fully connected layers end-to-end model object
    with the sequential API: https://keras.io/models/sequential/

    Parameters
    ----------
    verbose : bool
              if True prints the model summary (default True)

    Returns
    -------
    model : Sequential
            The instantiated model
    """
    model = Sequential()
    model.add(ConvLSTM2D(filters=64, kernel_size=(3,3), input_shape=(16,112,112,3)))
    model.add(Dropout(0.5))
    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(3, activation='softmax'))
    if verbose:
        model.summary()
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    return model

def get3DCNNLSTMModel(verbose=True, freeze_backbone=True, lr=1e-4):
    """Creates the 3DCNN + LSTM

    Parameters
    ----------
    verbose : bool
              if True prints the model summary (default True)

    Returns
    -------
    model : Sequential
            The instantiated model
    """
    pretrainedModel = getFeatureExtractor(BASE_DIR / "c3d_weight"/ "3dcnn_weights.h5", 'pool3', False)
    for layer in pretrainedModel.layers:
        layer.trainable = not freeze_backbone
        # layer.trainable = True

    # =========================
    # TimeDistributed(Flatten()) + LSTM
    # =========================
    # # (batch, 10, conv_filters)
    # x = TimeDistributed(Flatten())(pretrainedModel.output)
    # # (batch, lstm_units)
    # x = LSTM(64)(x)
    # =========================

    # or
    # =========================
    # ConvLSTM2D + BatchNormalization + GlobalAveragePooling2D
    # =========================
    # 2. Add ConvLSTM2D
    # We replace Flatten() and LSTM() with ConvLSTM2D
    # This keeps the (14, 14) spatial structure intact while processing the 4 time steps
    x = ConvLSTM2D(filters=64, kernel_size=(3, 3),
                   padding='same', return_sequences=False,
                   name='conv_lstm_layer')(pretrainedModel.output)

    # It's highly recommended to add BatchNormalization after ConvLSTM
    x = BatchNormalization()(x)

    # 3. Spatial Reduction
    # ConvLSTM2D (return_sequences=False) outputs (batch, 14, 14, 64)
    # Use GlobalAveragePooling2D to turn this into a vector of 64 features
    x = GlobalAveragePooling2D()(x)
    # =========================


    x = Dropout(.5)(x)
    # (batch, 256)
    x = Dense(256, activation='relu')(x)

    outputs = Dense(3, activation='softmax')(x)

    model = Model(pretrainedModel.inputs, outputs)
    if verbose:
        model.summary()

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr, beta_1=0.9, beta_2=0.999, epsilon=1e-08),
                    loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    del pretrainedModel

    return model

### 1. Streaming Videos Analysis

In [8]:
import os
from pathlib import Path
import cv2
import numpy as np
import tensorflow as tf


def run_real_world_video_inference(
    video_path,
    weights_path,
    getModel,
    class_names=None,
    freeze_backbone=True,
    lr=1e-4,
    batch_size=8,
    verbose=True
):
    """
    Now supports:
    - single video path
    - OR folder path (process all videos inside)
    """

    if class_names is None:
        class_names = ['non-violence', 'low-level violence', 'high-level violence']

    # -----------------------------
    # 1. Build model ONCE (important improvement)
    # -----------------------------
    model = getModel(verbose=verbose, freeze_backbone=freeze_backbone, lr=lr)
    model.load_weights(weights_path)

    # -----------------------------
    # 2. Detect if path is folder or file
    # -----------------------------
    video_path = Path(video_path)

    if video_path.is_dir():
        video_files = []
        for ext in ["*.mp4", "*.avi", "*.mov", "*.mkv"]:
            video_files.extend(video_path.glob(ext))
        video_files = sorted(video_files)
    else:
        video_files = [video_path]

    if len(video_files) == 0:
        print("No video files found.")
        return []

    all_results = []

    # -----------------------------
    # 3. Loop videos
    # -----------------------------
    for video_file in video_files:

        print(f"\n==============================")
        print(f"Processing: {video_file.name}")

        video = cv2.VideoCapture(str(video_file))
        if not video.isOpened():
            print(f"Cannot open video: {video_file}")
            continue

        numframes = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = int(video.get(cv2.CAP_PROP_FPS))
        chunks = numframes // 16

        if verbose:
            print(f"*** [Video Info] frames: {numframes} - fps: {fps} - chunks: {chunks}")

        videoFrames = []
        while True:
            ret, img = video.read()
            if not ret:
                break
            videoFrames.append(cv2.resize(img, (112, 112)))

        video.release()

        vid = np.array(videoFrames, dtype=np.float32)

        if chunks == 0:
            print("Too short, skip.")
            continue

        # -----------------------------
        # 4. Chunking (same as training)
        # -----------------------------
        X_list = []
        for i in range(chunks):
            X = vid[i * 16:i * 16 + 16]
            X_list.append(np.array(X, dtype=np.float32))

        X_infer = np.stack(X_list, axis=0).astype(np.float32)

        if verbose:
            print(f"Input shape: {X_infer.shape}")

        # -----------------------------
        # 5. Predict
        # -----------------------------
        chunk_probas = model.predict(X_infer, batch_size=batch_size, verbose=0)
        chunk_pred_ids = np.argmax(chunk_probas, axis=1)

        # -----------------------------
        # 6. Aggregate (same logic as your callback)
        # -----------------------------
        video_prob = np.mean(chunk_probas, axis=0)
        video_pred_id = int(np.argmax(video_prob))
        video_pred_label = class_names[video_pred_id]

        # -----------------------------
        # 7. Print result (what you want)
        # -----------------------------
        print(f"Video: {video_file.name}")
        print(f"Prediction: {video_pred_label}")
        print(f"Prob: {video_prob}")

        result = {
            "video_name": video_file.name,
            "video_path": str(video_file),
            "video_prob": video_prob,
            "video_pred_id": video_pred_id,
            "video_pred_label": video_pred_label
        }

        all_results.append(result)

    return all_results

In [ ]:
# =========================
# Example usage
# =========================

result = run_real_world_video_inference(
    video_path=BASE_DIR / "data" / "streaming videos" / "filter_results",
    # weights_path=r"weights/3DCNNLSTM.keras",
    weights_path=r"weights/C3DCNN.keras" ,
    # weights_path=r"weights/LSTM.keras",
    # getModel=get3DCNNLSTMModel,   # or getLSTMModel / getC3DCNNModel
    getModel=getC3DCNNModel,   
    # getModel=getLSTMModel,   
    class_names=['non-violence', 'low-level violence', 'high-level violence'],
    freeze_backbone=True,
    lr=1e-5,
    batch_size=8,
    verbose=True
)

Model: "functional_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ c3d_input (InputLayer)          │ (None, 16, 112, 112,   │             0 │
│                                 │ 3)                     │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv3D)                  │ (None, 16, 112, 112,   │         5,248 │
│                                 │ 64)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool1 (MaxPooling3D)            │ (None, 16, 56, 56, 64) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv3D)                  │ (None, 16, 56, 56,     │       221,312 │
│                                 │ 128)                   │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool2 (MaxPooling3D)            │ (None, 8, 28, 28, 128) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3a (Conv3D)                 │ (None, 8, 28, 28, 256) │       884,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3b (Conv3D)                 │ (None, 8, 28, 28, 256) │     1,769,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool3 (MaxPooling3D)            │ (None, 4, 14, 14, 256) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4a (Conv3D)                 │ (None, 4, 14, 14, 512) │     3,539,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv4b (Conv3D)                 │ (None, 4, 14, 14, 512) │     7,078,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool4 (MaxPooling3D)            │ (None, 2, 7, 7, 512)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5a (Conv3D)                 │ (None, 2, 7, 7, 512)   │     7,078,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv5b (Conv3D)                 │ (None, 2, 7, 7, 512)   │     7,078,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ zeropad5 (ZeroPadding3D)        │ (None, 2, 8, 8, 512)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ pool5 (MaxPooling3D)            │ (None, 1, 4, 4, 512)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc6 (Dense)                     │ (None, 4096)           │    33,558,528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc7-alt (Dense)                 │ (None, 512)            │     2,097,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 63,313,667 (241.52 MB)

 Trainable params: 2,099,203 (8.01 MB)

 Non-trainable params: 61,214,464 (233.51 MB)

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



Processing: Game_of_Thrones_S1_E8_(3.00-6.00_29.00-32.00)_sample_clip_1_00_20_to_00_25.mp4
*** [Video Info] frames: 172 - fps: 30 - chunks: 10
Input shape: (10, 16, 112, 112, 3)
Video: Game_of_Thrones_S1_E8_(3.00-6.00_29.00-32.00)_sample_clip_1_00_20_to_00_25.mp4
Prediction: non-violence
Prob: [0.5329125  0.4054087  0.06167882]

Processing: Game_of_Thrones_S1_E8_(3.00-6.00_29.00-32.00)_sample_clip_2_01_06_to_01_14.mp4
*** [Video Info] frames: 245 - fps: 30 - chunks: 15
Input shape: (15, 16, 112, 112, 3)
Video: Game_of_Thrones_S1_E8_(3.00-6.00_29.00-32.00)_sample_clip_2_01_06_to_01_14.mp4
Prediction: non-violence
Prob: [0.60455436 0.3163606  0.07908498]

Processing: Game_of_Thrones_S1_E8_(3.00-6.00_29.00-32.00)_sample_clip_3_01_35_to_02_21.mp4
*** [Video Info] frames: 1388 - fps: 30 - chunks: 86
Input shape: (86, 16, 112, 112, 3)
Video: Game_of_Thrones_S1_E8_(3.00-6.00_29.00-32.00)_sample_clip_3_01_35_to_02_21.mp4
Prediction: non-violence
Prob: [0.50995505 0.41771445 0.07233055]

Proce